# 06 - Modelos clásicos

## 1. Configuración y carga

In [1]:
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, GroupKFold, LeaveOneGroupOut, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
FEATURES = ["brillo", "contraste", "saturacion", "green_ratio", "edge_density"]
CLASES = ["bg", "challenge", "play", "throwin"]

MODELS_DIR = Path("../models")
RESULTS_DIR = Path("../results")
MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

frames = pd.read_csv("../data/processed/frame_metrics.csv")
X, y, groups = frames[FEATURES], frames["label"], frames["video_id"]
print(f"{len(frames)} frames de {groups.nunique()} videos")

9133 frames de 3 videos


## 2. Cómo se evalúa

Solo hay frames de 3 partidos. Cada modelo se entrena 3 veces, dejando cada vez un partido
completo fuera para probar (*leave-one-video-out*). Así el modelo nunca ve en prueba un partido
que ya vio entrenando.

Cuando un modelo busca sus mejores parámetros, la búsqueda usa solo los 2 partidos de
entrenamiento (entrena con uno y valida con el otro). El partido de prueba no participa.

In [2]:
def evaluar(nombre, pipeline):
    """Evalúa con leave-one-video-out, guarda el modelo final y devuelve (métricas, predicciones)."""
    filas_metricas, filas_preds = [], []
    es_busqueda = hasattr(pipeline, "param_grid") or hasattr(pipeline, "param_distributions")

    for train_idx, test_idx in LeaveOneGroupOut().split(X, y, groups):
        modelo = clone(pipeline)
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_te, y_te = X.iloc[test_idx], y.iloc[test_idx]

        t0 = time.perf_counter()
        if es_busqueda:
            modelo.fit(X_tr, y_tr, groups=groups.iloc[train_idx])
        else:
            modelo.fit(X_tr, y_tr)
        train_time = time.perf_counter() - t0
        fold = groups.iloc[test_idx].iloc[0]
        if es_busqueda:
            TUNING.append({"model": nombre, "fold": fold,
                           "best_params": modelo.best_params_,
                           "best_inner_f1": modelo.best_score_})

        t0 = time.perf_counter()
        pred = modelo.predict(X_te)
        proba_raw = modelo.predict_proba(X_te)
        pred_time_ms = (time.perf_counter() - t0) * 1000 / len(X_te)

        # columnas de probabilidad siempre en el orden de CLASES
        proba = pd.DataFrame(0.0, index=X_te.index, columns=CLASES)
        proba[list(modelo.classes_)] = proba_raw

        f1_clase = f1_score(y_te, pred, labels=CLASES, average=None, zero_division=0)
        filas_metricas.append({
            "model": nombre,
            "fold": fold,
            "accuracy": accuracy_score(y_te, pred),
            "f1_macro": f1_score(y_te, pred, labels=CLASES, average="macro", zero_division=0),
            **{f"f1_{c}": f for c, f in zip(CLASES, f1_clase)},
            "train_time_s": train_time,
            "pred_time_ms": pred_time_ms,
        })

        filas_preds.append(pd.DataFrame({
            "video_id": frames["video_id"].iloc[test_idx].values,
            "time_s": frames["time_s"].iloc[test_idx].values,
            "label": y_te.values,
            "model": nombre,
            "pred": pred,
            **{f"proba_{c}": proba[c].values for c in CLASES},
        }))

    # modelo final con todos los datos, es el que usa la app
    if es_busqueda:
        busqueda = clone(pipeline).fit(X, y, groups=groups)
        TUNING.append({"model": nombre, "fold": "final",
                       "best_params": busqueda.best_params_,
                       "best_inner_f1": busqueda.best_score_})
        final = busqueda.best_estimator_
    else:
        final = clone(pipeline).fit(X, y)
    joblib.dump(final, MODELS_DIR / f"{nombre}.joblib", compress=3)

    metricas = pd.DataFrame(filas_metricas)[[
        "model", "fold", "accuracy", "f1_macro", "f1_bg", "f1_play",
        "f1_challenge", "f1_throwin", "train_time_s", "pred_time_ms",
    ]]
    return metricas, pd.concat(filas_preds, ignore_index=True)

## 3. Modelo de referencia (Dummy)

Siempre predice la clase más común (`play`). Cualquier modelo real tiene que superarlo.

In [3]:
resultados = {}
TUNING = []  # parámetros elegidos por cada búsqueda

resultados["dummy"] = evaluar("dummy", Pipeline([
    ("clf", DummyClassifier(strategy="most_frequent")),
]))
resultados["dummy"][0]

,model,fold,accuracy,f1_macro,f1_bg,f1_play,f1_challenge,f1_throwin,train_time_s,pred_time_ms
0,dummy,1606b0e6_0,0.733678,0.211596,0.0,0.846383,0.0,0.0,0.001157,0.000040
1,dummy,1606b0e6_1,0.801492,0.222452,0.0,0.889809,0.0,0.0,0.000985,0.000023
2,dummy,35bd9041_0,0.803474,0.222757,0.0,0.891029,0.0,0.0,0.000995,0.000022


## 4. Regresión Logística

Necesita las variables en la misma escala. Se prueba cuánta regularización usar (`C`) y si
conviene darle más peso a las clases poco frecuentes (`class_weight`).

In [4]:
busqueda_logreg = GridSearchCV(
    Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ]),
    param_grid={"clf__C": [0.01, 0.1, 1, 10], "clf__class_weight": [None, "balanced"]},
    scoring="f1_macro", cv=GroupKFold(n_splits=2), n_jobs=-1,
)

resultados["logreg"] = evaluar("logreg", busqueda_logreg)
resultados["logreg"][0]

,model,fold,accuracy,f1_macro,f1_bg,f1_play,f1_challenge,f1_throwin,train_time_s,pred_time_ms
0,logreg,1606b0e6_0,0.733678,0.211596,0.00000,0.846383,0.0,0.0,1.441042,0.000393
1,logreg,1606b0e6_1,0.801492,0.222452,0.00000,0.889809,0.0,0.0,0.063237,0.000257
2,logreg,35bd9041_0,0.622956,0.227552,0.12731,0.782898,0.0,0.0,0.076148,0.000290


## 5. Random Forest

Muchos árboles de decisión que votan. Se prueban 20 combinaciones al azar de: cuántos árboles,
qué tan profundos, cuántos frames mínimo por hoja, cuántas variables mira cada corte y si se
balancean las clases.

In [5]:
busqueda_rf = RandomizedSearchCV(
    Pipeline([("clf", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))]),
    param_distributions={
        "clf__n_estimators": [100, 200, 300, 500],
        "clf__max_depth": [None, 5, 10, 20],
        "clf__min_samples_leaf": [1, 5, 20, 50],
        "clf__max_features": ["sqrt", "log2", None],
        "clf__class_weight": [None, "balanced", "balanced_subsample"],
    },
    n_iter=20, scoring="f1_macro", cv=GroupKFold(n_splits=2),
    random_state=RANDOM_STATE, n_jobs=1,  # el bosque ya usa todos los núcleos
)

resultados["rf"] = evaluar("rf", busqueda_rf)
resultados["rf"][0]

,model,fold,accuracy,f1_macro,f1_bg,f1_play,f1_challenge,f1_throwin,train_time_s,pred_time_ms
0,rf,1606b0e6_0,0.652158,0.218065,0.069597,0.802662,0.000000,0.0,10.937387,0.028746
1,rf,1606b0e6_1,0.749570,0.279938,0.229665,0.858342,0.031746,0.0,10.135390,0.007993
2,rf,35bd9041_0,0.802112,0.222548,0.000000,0.890191,0.000000,0.0,10.408741,0.009131


## 6. Guardar resultados

In [6]:
metrics = pd.concat([m for m, _ in resultados.values()], ignore_index=True)
preds = pd.concat([p for _, p in resultados.values()], ignore_index=True)

metrics.to_csv(RESULTS_DIR / "metrics_clasicos.csv", index=False)
preds.to_csv(RESULTS_DIR / "predictions_clasicos.csv", index=False)
pd.DataFrame(TUNING).to_csv(RESULTS_DIR / "tuning_clasicos.csv", index=False)
print(f"metrics: {metrics.shape}, predictions: {preds.shape}")

metrics: (9, 10), predictions: (27399, 9)


## 7. Resumen (promedio ± desviación de los 3 partidos)

In [7]:
cols = ["accuracy", "f1_macro", "f1_bg", "f1_play", "f1_challenge", "f1_throwin"]
resumen = metrics.groupby("model")[cols].agg(["mean", "std"]).round(3)
resumen

accuracy        f1_macro         f1_bg        f1_play         \
           mean    std     mean    std   mean    std    mean    std   
model                                                                 
dummy     0.780  0.040    0.219  0.006  0.000  0.000   0.876  0.025   
logreg    0.719  0.090    0.221  0.008  0.042  0.074   0.840  0.054   
rf        0.735  0.076    0.240  0.035  0.100  0.118   0.850  0.044   

       f1_challenge        f1_throwin       
               mean    std       mean  std  
model                                       
dummy         0.000  0.000        0.0  0.0  
logreg        0.000  0.000        0.0  0.0  
rf            0.011  0.018        0.0  0.0

In [8]:
pd.DataFrame(TUNING)

,model,fold,best_params,best_inner_f1
0,logreg,1606b0e6_0,"{'clf__C': 0.01, 'clf__class_weight': None}",0.222605
1,logreg,1606b0e6_1,"{'clf__C': 0.1, 'clf__class_weight': None}",0.218090
2,logreg,35bd9041_0,"{'clf__C': 10, 'clf__class_weight': None}",0.231947
3,logreg,final,"{'clf__C': 10, 'clf__class_weight': None}",0.227920
4,rf,1606b0e6_0,"{'clf__n_estimators': 500, 'clf__min_samples_l...",0.235706
5,rf,1606b0e6_1,"{'clf__n_estimators': 100, 'clf__min_samples_l...",0.234190
6,rf,35bd9041_0,"{'clf__n_estimators': 100, 'clf__min_samples_l...",0.254710
7,rf,final,"{'clf__n_estimators': 300, 'clf__min_samples_l...",0.255449


El Random Forest con búsqueda de parámetros es el mejor hasta ahora, pero por poco: sube el F1
macro de 0.219 (Dummy) a 0.240. La Regresión Logística queda igual que el Dummy, porque predice
casi siempre `play`. Ningún modelo reconoce `throwin`, y `challenge` casi nunca: con 5 propiedades
globales de la imagen no alcanza.